<a href="https://colab.research.google.com/github/BenjaminOrtizO/CaseStudy/blob/main/Entrega1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#CASE OF STUDY: BEBIDAS MAIPO S.A.
##Parte 1: Calidad y Pronóstico.
###Cargar el repositorio


In [19]:
!git clone https://github.com/BenjaminOrtizO/CaseStudy.git

# Entrar al repositorio
%cd /content/CaseStudy


!git pull

Cloning into 'CaseStudy'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 60 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 59.32 KiB | 906.00 KiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/CaseStudy
Already up to date.


###Cargar los datos


In [20]:
import pandas as pd
import numpy as np
import json

# Datos maestros
maestro_sku = pd.read_csv('Datos/maestro_sku.csv')
inventario_inicial = pd.read_csv('Datos/inventario_inicial.csv')

# Demanda y capacidad
ventas_semanal = pd.read_csv('Datos/demanda_semanal.csv')
capacidad_horizonte = pd.read_csv('Datos/capacidad_horizonte.csv')
compromisos_semana_focal = pd.read_csv('Datos/compromisos_semana_focal.csv')

# Calidad
calidad_atributos = pd.read_csv('Datos/calidad_atributos.csv')
calidad_llenado = pd.read_csv('Datos/calidad_llenado.csv')
especificaciones_calidad = pd.read_csv('Datos/especificaciones_calidad.csv')

# Configuración
setups_cip = pd.read_csv('Datos/setups_cip.csv')

# Metadatos
with open('Datos/metadatos.json', 'r', encoding='utf-8') as f:
    metadatos = json.load(f)

metadatos #equivalente a print en Jupyter

{'empresa': 'Bebidas del Maipo S.A.',
 'semilla': 20260923,
 'n_semanas_historia': 156,
 'fin_historia': '2026-09-21',
 'horizonte_planificacion': 13,
 'semana_focal_secuenciamiento': 12,
 'fecha_semana_focal': '2026-12-14',
 'n_sku': 9,
 'familias': ['CIT', 'FUN', 'NEC']}

##Preparación de los datos.


In [38]:
calidad_llenado.dtypes

,0
subgrupo,int64
fecha,object
semana_registro,int64
turno,object
linea,object
sku,object
formato_L,float64
hora,object
n_obs,int64
volumen_mL,float64


In [39]:
maestro_sku.dtypes

,0
sku,object
familia,object
nombre_familia,object
sabor,object
formato_L,float64
botellas_por_caja,int64
litros_por_caja,float64
linea_envasado,object
tasa_envasado_cajas_h,int64
costo_variable_caja_clp,int64


In [40]:
ventas_semanal.dtypes

,0
semana,int64
fecha_lunes,object
anio_iso,int64
semana_iso,int64
sku,object
ventas_cajas,float64
dias_sin_stock,float64
promocion,int64
precio_lista_clp,int64
precio_efectivo_clp,int64


In [41]:
calidad_atributos.dtypes

,0
fecha,object
semana_registro,int64
turno,object
linea,object
sku,object
n_inspeccionadas,int64
n_defectuosas,int64
causa_principal,object


In [42]:
especificaciones_calidad.dtypes

,0
formato_L,float64
nominal_mL,float64
LIE_mL,float64
LSE_mL,float64
tolerancia_oiml_mL,float64
limite_legal_inferior_mL,float64


In [45]:
calidad_llenado['fecha'] = pd.to_datetime(calidad_llenado['fecha'])
calidad_llenado['fecha'] = pd.to_datetime(calidad_llenado['fecha'])

In [21]:
calidad_llenado.info()
print('\n\n\n' , '='*300, '\n\n\n')
maestro_sku.info()
print('\n\n\n' , '='*300, '\n\n\n')
ventas_semanal.info()
print('\n\n\n' , '='*300, '\n\n\n')
calidad_atributos.info()
print('\n\n\n' , '='*300, '\n\n\n')
especificaciones_calidad.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   subgrupo         4200 non-null   int64  
 1   fecha            4200 non-null   object 
 2   semana_registro  4200 non-null   int64  
 3   turno            4200 non-null   object 
 4   linea            4200 non-null   object 
 5   sku              4200 non-null   object 
 6   formato_L        4200 non-null   float64
 7   hora             4200 non-null   object 
 8   n_obs            4200 non-null   int64  
 9   volumen_mL       4200 non-null   float64
dtypes: float64(2), int64(3), object(5)
memory usage: 328.3+ KB






<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 13 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   sku                               9 non

###Como se puede ver solo existen datos faltantes en el dataframe de ventas.

###En otras palabras, más adelante debemos tomar una decisión respecto a ello.

In [36]:
calidad_llenado.head(4200)

,subgrupo,fecha,semana_registro,turno,linea,sku,formato_L,hora,n_obs,volumen_mL
0,1,2026-07-20,1,A (06-14),L1,FUN-B05,0.5,06:00,1,500.86
1,1,2026-07-20,1,A (06-14),L1,FUN-B05,0.5,06:00,2,502.11
2,1,2026-07-20,1,A (06-14),L1,FUN-B05,0.5,06:00,3,503.12
3,1,2026-07-20,1,A (06-14),L1,FUN-B05,0.5,06:00,4,502.60
4,1,2026-07-20,1,A (06-14),L1,FUN-B05,0.5,06:00,5,499.11
...,...,...,...,...,...,...,...,...,...,...
4195,840,2026-09-25,10,C (22-06),L2,FUN-B15,1.5,04:00,1,1503.58
4196,840,2026-09-25,10,C (22-06),L2,FUN-B15,1.5,04:00,2,1504.76
4197,840,2026-09-25,10,C (22-06),L2,FUN-B15,1.5,04:00,3,1490.38
4198,840,2026-09-25,10,C (22-06),L2,FUN-B15,1.5,04:00,4,1506.28


In [35]:
maestro_sku.head(4200)

,sku,familia,nombre_familia,sabor,formato_L,botellas_por_caja,litros_por_caja,linea_envasado,tasa_envasado_cajas_h,costo_variable_caja_clp,precio_lista_caja_clp,costo_inventario_caja_semana_clp,penalidad_atraso_caja_semana_clp
0,NEC-D05,NEC,Nectares con pulpa,Durazno,0.5,12,6.0,L1,180,3840,6620,95,1400
1,NEC-D15,NEC,Nectares con pulpa,Durazno,1.5,6,9.0,L2,210,5760,9930,95,1400
2,NEC-D30,NEC,Nectares con pulpa,Durazno,3.0,4,12.0,L2,130,7680,13240,95,1400
3,CIT-L05,CIT,Citricos gasificados,Limon,0.5,12,6.0,L1,180,2820,5320,95,1400
4,CIT-L15,CIT,Citricos gasificados,Limon,1.5,6,9.0,L2,210,4230,7980,95,1400
5,CIT-L30,CIT,Citricos gasificados,Limon,3.0,4,12.0,L2,130,5640,10640,95,1400
6,FUN-B05,FUN,Funcionales,Berries,0.5,12,6.0,L1,180,4860,9920,95,1400
7,FUN-B15,FUN,Funcionales,Berries,1.5,6,9.0,L2,210,7290,14880,95,1400
8,FUN-B30,FUN,Funcionales,Berries,3.0,4,12.0,L2,130,9720,19840,95,1400


In [34]:
ventas_semanal.head(4200)

,semana,fecha_lunes,anio_iso,semana_iso,sku,ventas_cajas,dias_sin_stock,promocion,precio_lista_clp,precio_efectivo_clp
0,1,2023-10-02,2023,40,NEC-D05,1209.0,0.0,0,6620,6620
1,2,2023-10-09,2023,41,NEC-D05,1095.0,0.0,0,6620,6620
2,3,2023-10-16,2023,42,NEC-D05,1345.0,0.0,0,6620,6620
3,4,2023-10-23,2023,43,NEC-D05,1888.0,0.0,1,6620,4819
4,5,2023-10-30,2023,44,NEC-D05,991.0,0.0,0,6620,6620
...,...,...,...,...,...,...,...,...,...,...
1399,152,2026-08-24,2026,35,FUN-B30,0.0,0.0,1,19840,14445
1400,153,2026-08-31,2026,36,FUN-B30,0.0,0.0,0,19840,19840
1401,154,2026-09-07,2026,37,FUN-B30,0.0,0.0,0,19840,19840
1402,155,2026-09-14,2026,38,FUN-B30,324.0,3.0,0,19840,19840


In [33]:
calidad_atributos.head(4200)

,fecha,semana_registro,turno,linea,sku,n_inspeccionadas,n_defectuosas,causa_principal
0,2026-07-20,1,A (06-14),L1,FUN-B05,238,1,Envase deformado
1,2026-07-20,1,B (14-22),L1,CIT-L05,182,3,Etiqueta
2,2026-07-21,1,A (06-14),L1,NEC-D05,190,4,Envase deformado
3,2026-07-21,1,B (14-22),L1,CIT-L05,194,3,Nivel de llenado
4,2026-07-22,1,A (06-14),L1,CIT-L05,190,8,Etiqueta
...,...,...,...,...,...,...,...,...
205,2026-09-24,10,B (14-22),L2,CIT-L30,196,6,Etiqueta
206,2026-09-24,10,C (22-06),L2,NEC-D15,213,6,Etiqueta
207,2026-09-25,10,A (06-14),L2,CIT-L30,218,5,Sello/tapa
208,2026-09-25,10,B (14-22),L2,CIT-L30,244,6,Sello/tapa


In [32]:
especificaciones_calidad.head(4200)

,formato_L,nominal_mL,LIE_mL,LSE_mL,tolerancia_oiml_mL,limite_legal_inferior_mL
0,0.5,500.0,492.0,508.0,15.0,485.0
1,1.5,1500.0,1485.0,1515.0,22.5,1477.5
2,3.0,3000.0,2975.0,3025.0,45.0,2955.0


##Los datos no permiten separar dos fuentes de variación, pues no se pueden identificar operadores ni instrumentos. Además, tampoco se ve que existan repeticiones.

##Por lo tanto, se procederá a establecer los límites.

#ETAPA A

##DATOS A UTILIZAR
###-calidad_llenado
###-calidad_atributos
###-especificaciones_calidad
###-maestro_sku
###-ventas_semanal


##Fase 1
###Establecer Límites.
####Previo a establecer los límites se debe saber si el instrumento está midiendo de manera apropiada. Para ello se debe realizar un análisis de MSA.